# Build a Tiny GPT, End to End, and Train It on This Blog

The runnable code of [Build a Tiny GPT, End to End, and Train It on This Blog](https://residual-stream.lvtan71.workers.dev/posts/tiny-gpt-from-scratch/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/tiny-gpt-from-scratch.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 314 min end to end.
Trains about twenty small models, several hours on a T4. The baseline and cosine-schedule runs reproduce the post exactly; the ablations (no position, learned positions, no residual) can differ in the third decimal.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "tiny-gpt-from-scratch"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/tiny-gpt-from-scratch/datafrac.py', 'shared/tiny-gpt-from-scratch/final.py', 'shared/tiny-gpt-from-scratch/generate.py', 'shared/tiny-gpt-from-scratch/gpt.py', 'shared/tiny-gpt-from-scratch/kn.py', 'shared/autoregression-and-perplexity/lm.py', 'shared/tiny-gpt-from-scratch/repeats.py', 'shared/tiny-gpt-from-scratch/train.py', 'data/blog_train.txt', 'data/blog_val.txt']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## The model

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/tiny-gpt-from-scratch/#the-model)

In [ ]:
import math
import torch
import torch.nn.functional as F

# A tiny GPT, every piece from the earlier posts: token embeddings ("Embeddings and Contrastive Learning"),
# RoPE ("Where Am I?"), pre-norm RMSNorm ("Dropout and Normalization"), causal multi-head attention
# ("The Transformer's Three Attention Patterns"), a SwiGLU MLP, residual connections ("The Residual Stream"), weight-tied
# unembedding, trained on next-token cross-entropy ("Cross-Entropy and the Softmax Gradient", "What 'Generative' Means").
class Config:
    def __init__(self, **kw):
        self.V, self.T, self.D, self.L, self.N = 2048, 256, 192, 4, 4     # vocab, context, width, layers, heads
        self.dropout, self.position, self.residual, self.prenorm = 0.1, "rope", True, True
        self.__dict__.update(kw)

def rope(x):                                          # x: (B, heads, T, H)
    T, H = x.shape[-2], x.shape[-1]
    ang = torch.arange(T, device=x.device)[:, None] * 10_000.0 ** (-torch.arange(0, H, 2, device=x.device) / H)
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * ang.cos() - x2 * ang.sin(), x1 * ang.sin() + x2 * ang.cos()], -1).flatten(-2)

class RMSNorm(torch.nn.Module):
    def __init__(self, D):
        super().__init__(); self.g = torch.nn.Parameter(torch.ones(D))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6) * self.g

class Block(torch.nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c, F_ = c, int(8 * c.D / 3)              # SwiGLU hidden width: 8/3 D keeps the MLP at 8 D^2 weights
        self.n1, self.n2 = RMSNorm(c.D), RMSNorm(c.D)
        self.qkv, self.proj = torch.nn.Linear(c.D, 3 * c.D, bias=False), torch.nn.Linear(c.D, c.D, bias=False)
        self.w_gate, self.w_up = torch.nn.Linear(c.D, F_, bias=False), torch.nn.Linear(c.D, F_, bias=False)
        self.w_down = torch.nn.Linear(F_, c.D, bias=False)
        self.drop = torch.nn.Dropout(c.dropout)
    def attention(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.c.N, D // self.c.N).permute(2, 0, 3, 1, 4)
        if self.c.position == "rope":
            q, k = rope(q), rope(k)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.c.dropout if self.training else 0)
        return self.drop(self.proj(y.transpose(1, 2).reshape(B, T, D)))
    def mlp(self, x):
        return self.drop(self.w_down(F.silu(self.w_gate(x)) * self.w_up(x)))
    def forward(self, x):
        c = self.c
        if not c.prenorm:                              # the 2017 layout: add, then normalize
            x = self.n1(x + self.attention(x))
            return self.n2(x + self.mlp(x))
        if not c.residual:                             # no shortcut: each sublayer replaces the stream
            x = self.attention(self.n1(x))
            return self.mlp(self.n2(x))
        x = x + self.attention(self.n1(x))
        return x + self.mlp(self.n2(x))

class GPT(torch.nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.embed = torch.nn.Embedding(c.V, c.D)
        self.pos = torch.nn.Embedding(c.T, c.D) if c.position == "learned" else None
        self.blocks = torch.nn.ModuleList([Block(c) for _ in range(c.L)])
        self.norm = RMSNorm(c.D)
        self.apply(self._init)
    def _init(self, m):
        if isinstance(m, (torch.nn.Linear, torch.nn.Embedding)):
            torch.nn.init.normal_(m.weight, 0, 0.02)
    def forward(self, idx):
        x = self.embed(idx)
        if self.pos is not None:
            x = x + self.pos(torch.arange(idx.shape[1], device=idx.device))
        for b in self.blocks:
            x = b(x)
        return self.norm(x) @ self.embed.weight.T          # unembedding shares the embedding table

## Counting the parameters

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/tiny-gpt-from-scratch/#counting-the-parameters)

In [ ]:
from gpt import GPT, Config

# Where the parameters are, counted from the model and from the formula:
# per block 4 D^2 in attention (Q, K, V, output) and 8 D^2 in the SwiGLU MLP,
# plus the embedding table, which the unembedding shares.
c = Config(V=148)                                     # the character model trained below
model = GPT(c)
groups = {"embedding (shared with unembedding)": [model.embed],
          "attention, all blocks": [m for b in model.blocks for m in (b.qkv, b.proj)],
          "MLP, all blocks": [m for b in model.blocks for m in (b.w_gate, b.w_up, b.w_down)],
          "norm gains": [n for b in model.blocks for n in (b.n1, b.n2)] + [model.norm]}
total = sum(p.numel() for p in model.parameters())
for name, mods in groups.items():
    n = sum(p.numel() for m in mods for p in m.parameters())
    print(f"{name:37s} {n:>9,}  {n / total:5.1%}")
print(f"{'total':37s} {total:>9,}")
formula = lambda L, D, V: 12 * L * D ** 2 + V * D
print(f"formula 12 L D^2 + V D: {formula(c.L, c.D, c.V):,}  (L={c.L}, D={c.D}, V={c.V})")
print(f"with post 9's 2,048 BPE tokens instead: {formula(4, 192, 2048):,}, "
      f"embedding {2048 * 192 / formula(4, 192, 2048):.0%} of it")
print(f"GPT-2 small (L=12, D=768, V=50,257): {formula(12, 768, 50257) / 1e6:.1f}M by the formula; "
      f"124M reported, the rest mostly its 1,024 learned position vectors")

The post printed:

```text
embedding (shared with unembedding)      28,416   1.6%
attention, all blocks                   589,824  32.8%
MLP, all blocks                       1,179,648  65.5%
norm gains                                1,728   0.1%
total                                 1,799,616
formula 12 L D^2 + V D: 1,797,888  (L=4, D=192, V=148)
with post 9's 2,048 BPE tokens instead: 2,162,688, embedding 18% of it
GPT-2 small (L=12, D=768, V=50,257): 123.5M by the formula; 124M reported, the rest mostly its 1,024 learned position vectors
```

## Training, with the choices made on a dev slice

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/tiny-gpt-from-scratch/#training-with-the-choices-made-on-a-dev-slice)

In [ ]:
%%writefile train.py
import json
import math
import sys
import torch
import torch.nn.functional as F
from tokenizers import Tokenizer, models, pre_tokenizers, trainers
from gpt import GPT, Config
GPU = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"  # notebook: the post ran on "mps"

# Train the tiny GPT on the blog. Hyperparameters are chosen on a dev slice,
# the last 10% of the training text; the two held-out posts are scored once,
# at the end, with the choice already made.
full_train, held_out = open("blog_train.txt").read(), open("blog_val.txt").read()
cut = int(0.9 * len(full_train))
dev_split = {"fit": full_train[:cut], "score": full_train[cut:]}     # choose here
final_split = {"fit": full_train, "score": held_out}               # report here

def tokenizer_for(kind, text):
    if kind == "char":
        vocab = sorted(set(text))
        index = {c: i + 1 for i, c in enumerate(vocab)}                 # unseen characters share id 0
        return len(vocab) + 1, lambda s: torch.tensor([index.get(c, 0) for c in s]), vocab
    bpe = Tokenizer(models.BPE())
    bpe.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    bpe.train_from_iterator([text], trainers.BpeTrainer(
        vocab_size=2048, initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False))
    return bpe.get_vocab_size(), lambda s: torch.tensor(bpe.encode(s).ids), bpe

@torch.no_grad()
def bits_per_byte(model, data, text, T):
    """Every token scored once, each with at least T/2 tokens of context
    (after the first window): windows of T, stepped by T/2, scoring the
    second half of each."""
    model.eval()
    total, s = 0.0, 0
    while True:
        x = data[s:s + T + 1]
        if len(x) < 2:
            break
        logits = model(x[None, :-1])[0]
        first = 0 if s == 0 else T // 2
        total += F.cross_entropy(logits[first:], x[1 + first:], reduction="sum").item()
        if s + T + 1 >= len(data):
            break
        s += T // 2
    model.train()
    return total / math.log(2) / len(text.encode("utf-8"))

def run(kind, seed, split, steps, checkpoints=(), B=32, lr=1e-3, warmup=200, schedule="constant",
        device=GPU, on_checkpoint=None, **cfg):
    V, encode, _ = tokenizer_for(kind, split["fit"])
    fit, score = encode(split["fit"]).to(device), encode(split["score"]).to(device)
    c = Config(V=V, **cfg)
    torch.manual_seed(seed)
    model = GPT(c).to(device)
    decay = [p for p in model.parameters() if p.dim() == 2]            # matrices decay; norm gains don't
    rest = [p for p in model.parameters() if p.dim() < 2]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": 0.1}, {"params": rest, "weight_decay": 0.0}],
                            lr=lr, betas=(0.9, 0.95))
    g = torch.Generator().manual_seed(seed)
    curve = {}
    for step in range(1, steps + 1):
        frac = min(1.0, step / warmup)
        if schedule == "cosine" and step > warmup:
            frac = 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * (step - warmup) / (steps - warmup)))
        for group in opt.param_groups:
            group["lr"] = lr * frac
        i = torch.randint(0, len(fit) - c.T - 1, (B,), generator=g).to(device)
        idx = i[:, None] + torch.arange(c.T + 1, device=device)
        x, y = fit[idx[:, :-1]], fit[idx[:, 1:]]
        loss = F.cross_entropy(model(x).reshape(-1, V), y.reshape(-1))
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if step in checkpoints or step == steps:
            curve[step] = bits_per_byte(model, score, split["score"], c.T)
            if on_checkpoint:
                on_checkpoint(step, model)
    passes = steps * B * c.T / len(fit)
    return model, curve, passes

if __name__ == "__main__":
    # python train.py <name> '<json of run() keyword arguments>'
    name, kw = sys.argv[1], json.loads(sys.argv[2])
    split = final_split if kw.pop("final", False) else dev_split
    save = kw.pop("save", None)
    model, curve, passes = run(split=split, **kw)
    if save:
        cfg = {k: v for k, v in model.c.__dict__.items()}
        vocab = tokenizer_for(kw["kind"], split["fit"])[2] if kw["kind"] == "char" else None
        torch.save({"model": model.cpu().state_dict(), "config": cfg, "vocab": vocab}, save)
    print(f"{name}: " + " / ".join(f"{s}: {v:.3f}" for s, v in curve.items()) + f"  ({passes:.0f} passes)",
          flush=True)

In [ ]:
# How the post ran train.py: as a script, one configuration per run.
import json, subprocess
def train(name, **kw):
    out = subprocess.run([sys.executable, "train.py", name, json.dumps(kw)], capture_output=True, text=True)
    print(out.stdout.strip() if out.returncode == 0 else out.stderr[-2000:], flush=True)

CK = [250, 500, 750, 1000, 1500, 2000, 2500, 3000]
train("char d0.1 s0", kind="char", seed=0, steps=3000, checkpoints=CK)
train("bpe  d0.1 s0", kind="bpe", seed=0, steps=3000, checkpoints=CK)
train("char d0.2 s0", kind="char", seed=0, steps=3000, dropout=0.2, checkpoints=CK)
train("bpe  d0.2 s0", kind="bpe", seed=0, steps=3000, dropout=0.2, checkpoints=CK)
CKC, CKB = [1000, 2000, 3000, 4000, 5000, 6000], [250, 500, 750, 1000, 1250, 1500, 2000]
train("char d0.2 s0", kind="char", seed=0, steps=6000, dropout=0.2, checkpoints=CKC)
train("char d0.3 s0", kind="char", seed=0, steps=6000, dropout=0.3, checkpoints=CKC)
train("bpe  d0.3 s0", kind="bpe", seed=0, steps=2000, dropout=0.3, checkpoints=CKB)
train("bpe  d0.4 s0", kind="bpe", seed=0, steps=2000, dropout=0.4, checkpoints=CKB)

The post printed:

```text
char d0.1 s0: 250: 2.741 / 500: 2.273 / 750: 2.158 / 1000: 2.076 / 1500: 2.080 / 2000: 2.072 / 2500: 2.105 / 3000: 2.141  (46 passes)
bpe  d0.1 s0: 250: 2.415 / 500: 2.116 / 750: 2.204 / 1000: 2.396 / 1500: 2.791 / 2000: 3.066 / 2500: 3.218 / 3000: 3.333  (126 passes)
char d0.2 s0: 250: 2.919 / 500: 2.398 / 750: 2.230 / 1000: 2.147 / 1500: 2.093 / 2000: 2.053 / 2500: 2.040 / 3000: 2.034  (46 passes)
bpe  d0.2 s0: 250: 2.461 / 500: 2.129 / 750: 2.111 / 1000: 2.177 / 1500: 2.394 / 2000: 2.547 / 2500: 2.693 / 3000: 2.789  (126 passes)
char d0.2 s0: 1000: 2.147 / 2000: 2.053 / 3000: 2.034 / 4000: 2.027 / 5000: 2.029 / 6000: 2.038  (92 passes)
char d0.3 s0: 1000: 2.254 / 2000: 2.116 / 3000: 2.088 / 4000: 2.030 / 5000: 2.039 / 6000: 2.015  (92 passes)
bpe  d0.3 s0: 250: 2.522 / 500: 2.159 / 750: 2.106 / 1000: 2.114 / 1250: 2.169 / 1500: 2.201 / 2000: 2.263  (84 passes)
bpe  d0.4 s0: 250: 2.603 / 500: 2.216 / 750: 2.128 / 1000: 2.117 / 1250: 2.119 / 1500: 2.122 / 2000: 2.164  (84 passes)
```

## Taking pieces out

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/tiny-gpt-from-scratch/#taking-pieces-out)

In [ ]:
# The shell loop above, in Python:
BASE = dict(kind="char", steps=4000, dropout=0.2, checkpoints=[1000, 2000, 3000, 4000])
for seed in (0, 1):
    train(f"baseline        s{seed}", **BASE, seed=seed)
    train(f"cosine schedule s{seed}", **BASE, seed=seed, schedule="cosine")
    train(f"no position     s{seed}", **BASE, seed=seed, position="none")
    train(f"learned pos.    s{seed}", **BASE, seed=seed, position="learned")
    train(f"post-norm       s{seed}", **BASE, seed=seed, prenorm=False)
    train(f"no residual     s{seed}", **BASE, seed=seed, residual=False)

The post printed:

```text
baseline        s0: 1000: 2.147 / 2000: 2.053 / 3000: 2.034 / 4000: 2.027  (61 passes)
cosine schedule s0: 1000: 2.141 / 2000: 2.016 / 3000: 2.001 / 4000: 1.990  (61 passes)
no position     s0: 1000: 3.318 / 2000: 2.954 / 3000: 2.779 / 4000: 2.706  (61 passes)
learned pos.    s0: 1000: 2.560 / 2000: 2.260 / 3000: 2.176 / 4000: 2.116  (61 passes)
post-norm       s0: 1000: 2.292 / 2000: 2.127 / 3000: 2.095 / 4000: 2.088  (61 passes)
no residual     s0: 1000: 5.061 / 2000: 5.022 / 3000: 5.049 / 4000: 5.048  (61 passes)
baseline        s1: 1000: 2.140 / 2000: 2.030 / 3000: 2.023 / 4000: 2.006  (61 passes)
cosine schedule s1: 1000: 2.134 / 2000: 2.012 / 3000: 1.992 / 4000: 1.982  (61 passes)
no position     s1: 1000: 3.374 / 2000: 2.918 / 3000: 2.790 / 4000: 2.688  (61 passes)
learned pos.    s1: 1000: 2.563 / 2000: 2.281 / 3000: 2.169 / 4000: 2.111  (61 passes)
post-norm       s1: 1000: 2.301 / 2000: 2.119 / 3000: 2.101 / 4000: 2.083  (61 passes)
no residual     s1: 1000: 5.037 / 2000: 5.051 / 3000: 5.048 / 4000: 5.043  (61 passes)
```

## What happens without the residual stream

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/tiny-gpt-from-scratch/#what-happens-without-the-residual-stream)

In [ ]:
import math
import torch
import torch.nn.functional as F
from gpt import GPT, Config
from train import dev_split, tokenizer_for

# Why does the model without residual connections stop at the unigram level?
# Train loss, and the gradient reaching each layer at the first steps.
torch.set_num_threads(4)
V, encode, _ = tokenizer_for("char", dev_split["fit"])
fit = encode(dev_split["fit"])
counts = torch.bincount(fit, minlength=V).float()
p = counts / counts.sum()
print(f"unigram entropy of the training text: {-(p[p > 0] * p[p > 0].log2()).sum():.3f} bits per character")
for residual in (True, False):
    torch.manual_seed(0); g = torch.Generator().manual_seed(0)
    m = GPT(Config(V=V, residual=residual, dropout=0.2))
    opt = torch.optim.AdamW(m.parameters(), 1e-3, betas=(0.9, 0.95))
    for step in range(1, 401):
        for gr in opt.param_groups:
            gr["lr"] = 1e-3 * min(1, step / 200)
        i = torch.randint(0, len(fit) - 257, (16,), generator=g)
        idx = i[:, None] + torch.arange(257)
        x, y = fit[idx[:, :-1]], fit[idx[:, 1:]]
        logits = m(x)
        loss = F.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
        opt.zero_grad(); loss.backward()
        if step in (1, 400):
            norms = [m.blocks[l].qkv.weight.grad.norm().item() for l in range(4)]
            print(f"residual={residual!s:5} step {step:3d}: train loss {loss.item() / math.log(2):.3f} bits; "
                  f"gradient norm at the attention weights of layers 1-4: "
                  + " ".join(f"{n:.1e}" for n in norms))
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        opt.step()
    with torch.no_grad():                       # does the output depend on the input at all?
        m.eval()
        a, b = m(x[:1])[0, -1].softmax(-1), m(x[1:2])[0, -1].softmax(-1)
        print(f"   output distributions for two different contexts differ by {0.5 * (a - b).abs().sum():.3f} (total variation)")

The post printed:

```text
unigram entropy of the training text: 5.015 bits per character
residual=True  step   1: train loss 7.214 bits; gradient norm at the attention weights of layers 1-4: 2.0e+00 1.7e+00 1.2e+00 9.3e-01
residual=True  step 400: train loss 2.621 bits; gradient norm at the attention weights of layers 1-4: 5.6e-01 1.5e-01 1.5e-01 1.7e-01
   output distributions for two different contexts differ by 0.861 (total variation)
residual=False step   1: train loss 7.246 bits; gradient norm at the attention weights of layers 1-4: 2.0e+00 1.4e+00 1.0e+00 7.5e-01
residual=False step 400: train loss 5.086 bits; gradient norm at the attention weights of layers 1-4: 1.6e-06 1.0e-06 3.3e-06 1.1e-04
   output distributions for two different contexts differ by 0.000 (total variation)
```

In [ ]:
# The shell loop above, in Python:
for seed in (0, 1):
    train(f"no residual, lr 1e-4 s{seed}", **BASE, seed=seed, residual=False, lr=0.0001)
for seed in (0, 1):
    train(f"baseline,    lr 1e-4 s{seed}", **BASE, seed=seed, lr=0.0001)

The post printed:

```text
no residual, lr 1e-4 s0: 1000: 4.400 / 2000: 3.733 / 3000: 3.398 / 4000: 3.122  (61 passes)
no residual, lr 1e-4 s1: 1000: 4.373 / 2000: 3.660 / 3000: 3.328 / 4000: 3.041  (61 passes)
baseline,    lr 1e-4 s0: 1000: 2.842 / 2000: 2.374 / 3000: 2.212 / 4000: 2.119  (61 passes)
baseline,    lr 1e-4 s1: 1000: 2.837 / 2000: 2.337 / 3000: 2.169 / 4000: 2.102  (61 passes)
```

## The score, once

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/tiny-gpt-from-scratch/#the-score-once)

In [ ]:
%%writefile final.py
import sys
import torch
from train import run, final_split, tokenizer_for

# The one scored run: the settings chosen on the dev slice, trained on all
# the training posts, scored on the two held-out posts. Saves the model at
# a few checkpoints for sampling.
seed, schedule = int(sys.argv[1]), sys.argv[2]
vocab = tokenizer_for("char", final_split["fit"])[2]
def save(step, model):
    if seed == 0:
        state = {k: v.cpu() for k, v in model.state_dict().items()}
        torch.save({"model": state, "config": dict(model.c.__dict__), "vocab": vocab}, f"gpt_{step}.pt")
_, curve, passes = run("char", seed, final_split, 4000, checkpoints=(250, 1000, 2000, 3000),
                       dropout=0.2, schedule=schedule, on_checkpoint=save)
print(f"seed {seed}, {schedule} schedule, held-out bits per byte at "
      + " / ".join(f"{s:,}: {v:.3f}" for s, v in curve.items()) + f" steps ({passes:.0f} passes)")

In [ ]:
# The post's final runs: three seeds, cosine schedule (seed 0 saves the checkpoints).
for seed in (0, 1, 2):
    %run final.py {seed} cosine

The post printed:

```text
seed 0, cosine schedule, held-out bits per byte at 250: 3.036 / 1,000: 2.511 / 2,000: 2.453 / 3,000: 2.475 / 4,000: 2.496 steps (55 passes)
seed 1, cosine schedule, held-out bits per byte at 250: 3.005 / 1,000: 2.607 / 2,000: 2.491 / 3,000: 2.562 / 4,000: 2.633 steps (55 passes)
seed 2, cosine schedule, held-out bits per byte at 250: 3.019 / 1,000: 2.492 / 2,000: 2.436 / 3,000: 2.570 / 4,000: 2.535 steps (55 passes)
```

**Not shown in the post:** this comparison includes post 9's character-level LSTM, which post 9's `lm.py` saves as `char_lstm.pt` when it trains on characters with seed 0. This trains it again (a few minutes).

In [ ]:
%run lm.py char 0

In [ ]:
import bz2
import math
import sys
import torch
import torch.nn.functional as F
from gpt import GPT, Config
from repeats import bits_each
from train import final_split, tokenizer_for
from kn import build, prob                        # Kneser-Ney from "Sequences Before Attention", unchanged
sys.path.insert(0, "../post9")
from lm import LSTMLM                              # LSTM from "What 'Generative' Means"

# The two held-out posts scored separately, by every model on the ladder in
# "What 'Generative' Means", plus the GPT's score on training text it has seen, at each saved
# checkpoint (seed 0). Each model scores the whole held-out text as one
# piece, as before; the per-character costs are then summed per post.
torch.set_num_threads(4)
train, val = final_split["fit"], final_split["score"]
cut = val.find("When an LLM generates text")       # the KV-cache post starts here
posts = {"BPE post": (1, cut), "KV-cache post": (cut, len(val))}
nbytes = lambda s: len(s.encode("utf-8"))
_, encode, vocab = tokenizer_for("char", train)

def per_post(bits):                                 # bits[i] is the cost of character i
    return {name: bits[a:b].sum().item() / nbytes(val[a:b]) for name, (a, b) in posts.items()}

unseen = torch.tensor([c not in set(train) for c in val])
print(f"{unseen.sum()} held-out characters never occur in training: "
      + ", ".join(f"{name} {unseen[a:b].sum()}" for name, (a, b) in posts.items()))

rows = {}
for step in (250, 1000, 2000, 3000, 4000):
    ckpt = torch.load(f"gpt_{step}.pt", weights_only=False)
    model = GPT(Config(**ckpt["config"])); model.load_state_dict(ckpt["model"]); model.eval()
    bits = torch.cat([torch.zeros(1), bits_each(model, encode(val), model.c.T)])
    seen_text = train[:60_000]
    train_bpb = bits_each(model, encode(seen_text), model.c.T).sum().item() / nbytes(seen_text[1:])
    rows[f"GPT, {step:,} steps"] = (per_post(bits), train_bpb)
    if step == 4000:
        cost = bits[unseen].sum().item()
        print(f"the GPT at 4,000 steps pays {cost / unseen.sum():.1f} bits per unseen character, "
              f"{cost / nbytes(val):.3f} bits per byte of the held-out text in all")

lstm = torch.load("char_lstm.pt", weights_only=False)
index = {u: i + 1 for i, u in enumerate(lstm["vocab"])}
m = LSTMLM(len(lstm["vocab"]) + 1); m.load_state_dict(lstm["model"]); m.eval()
data, bits, state = torch.tensor([index.get(c, 0) for c in val]), torch.zeros(len(val)), None
with torch.no_grad():
    for s in range(0, len(data) - 1, 128):
        x, y = data[s:s + 128], data[s + 1:s + 129]
        logits, state = m(x[None][:, :len(y)], state)
        bits[s + 1:s + 1 + len(y)] = F.cross_entropy(logits[0], y, reduction="none") / math.log(2)
rows["LSTM (post 9, seed 0)"] = (per_post(bits), None)

tables = build(6)
bits = torch.tensor([0.0] + [-math.log2(prob(tables, val[max(0, i - 6):i], val[i])) for i in range(1, len(val))])
rows["Kneser-Ney, 6 characters"] = (per_post(bits), None)

tb = train.encode()
rows["bz2 given the training posts"] = ({name: 8 * (len(bz2.compress(tb + val[a:b].encode(), 9)) - len(bz2.compress(tb, 9)))
                                         / nbytes(val[a:b]) for name, (a, b) in posts.items()}, None)

print(f"\n{'bits per byte':30s} {'BPE post':>9s} {'KV-cache post':>14s} {'training text (seen)':>21s}")
for name, (pp, tr) in rows.items():
    print(f"{name:30s} {pp['BPE post']:9.3f} {pp['KV-cache post']:14.3f} "
          + (f"{tr:21.3f}" if tr is not None else ""))

The post printed:

```text
123 held-out characters never occur in training: BPE post 122, KV-cache post 1
the GPT at 4,000 steps pays 25.6 bits per unseen character, 0.055 bits per byte of the held-out text in all

bits per byte                   BPE post  KV-cache post  training text (seen)
GPT, 250 steps                     3.212          2.654                 2.698
GPT, 1,000 steps                   2.775          1.935                 1.803
GPT, 2,000 steps                   2.747          1.815                 1.566
GPT, 3,000 steps                   2.801          1.763                 1.398
GPT, 4,000 steps                   2.842          1.744                 1.327
LSTM (post 9, seed 0)              2.944          2.033 
Kneser-Ney, 6 characters           2.826          2.003 
bz2 given the training posts       2.480          2.038 
```

## Why bz2 wins the BPE post

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/tiny-gpt-from-scratch/#why-bz2-wins-the-bpe-post)

In [ ]:
import math
import torch
import torch.nn.functional as F
from gpt import GPT, Config
from train import dev_split, final_split, tokenizer_for

# How much of the held-out text repeats something, and the model's loss on
# it. A character counts as "copied" if it lies inside a stretch of SPAN or
# more characters that also appears word for word in the training text, and
# as "repeated" if that stretch appeared earlier in the held-out text itself,
# more than 256 characters back, beyond the model's window.
torch.set_num_threads(4)

def copied_mask(text, train, span):
    covered = torch.zeros(len(text), dtype=torch.bool)
    for i in range(len(text) - span + 1):
        if text[i:i + span] in train:
            covered[i:i + span] = True
    return covered

def repeated_mask(text, span, T=256):
    covered, first_seen = torch.zeros(len(text), dtype=torch.bool), {}
    for i in range(len(text) - span + 1):
        s = text[i:i + span]
        if s in first_seen and first_seen[s] < i - T:
            covered[i:i + span] = True
        first_seen.setdefault(s, i)
    return covered

@torch.no_grad()
def bits_each(model, data, T):
    """Bits for each character from the second on, scored as in train.py."""
    out = torch.zeros(len(data) - 1)
    s = 0
    while True:
        x = data[s:s + T + 1]
        logits = model(x[None, :-1])[0]
        first = 0 if s == 0 else T // 2
        out[s + first:s + len(x) - 1] = F.cross_entropy(logits[first:], x[1 + first:], reduction="none") / math.log(2)
        if s + T + 1 >= len(data):
            break
        s += T // 2
    return out

if __name__ == "__main__":
    ckpt = torch.load("gpt_4000.pt", weights_only=False)
    model = GPT(Config(**ckpt["config"])); model.load_state_dict(ckpt["model"]); model.eval()
    train, text = final_split["fit"], final_split["score"]
    _, encode, _ = tokenizer_for("char", train)
    bits = bits_each(model, encode(text), model.c.T)             # bits per character, characters 1 onward
    print(f"the model on the held-out posts: {bits.mean():.3f} bits per character")
    for span in (32, 16):
        copied = copied_mask(text, train, span)[1:]
        rep = repeated_mask(text, span)[1:] & ~copied
        rest = ~copied & ~rep
        print(f"stretches of {span}+ characters: copied {copied.float().mean():5.1%} at {bits[copied].mean():.2f} bits, "
              f"repeated {rep.float().mean():5.1%} at {bits[rep].mean():.2f}, the rest {rest.float().mean():5.1%} "
              f"at {bits[rest].mean():.2f}")
    dev_copied = copied_mask(dev_split["score"], dev_split["fit"], 32).float().mean()
    print(f"for comparison, the dev slice: {dev_copied:.1%} copied from the rest of the training text (32+)")

The post printed:

```text
the model on the held-out posts: 2.522 bits per character
stretches of 32+ characters: copied  3.8% at 0.58 bits, repeated  1.6% at 5.15, the rest 94.7% at 2.56
stretches of 16+ characters: copied 15.4% at 1.07 bits, repeated  6.2% at 3.45, the rest 78.4% at 2.73
for comparison, the dev slice: 5.6% copied from the rest of the training text (32+)
```

In [ ]:
%%writefile datafrac.py
import sys
from train import run, full_train, held_out

# The same model and schedule, trained on the first quarter or half of the
# training text, scored on the held-out posts: how much does more data help?
frac, seed = float(sys.argv[1]), int(sys.argv[2])
split = {"fit": full_train[:int(frac * len(full_train))], "score": held_out}
_, curve, passes = run("char", seed, split, 4000, checkpoints=(250, 500, 750, 1000, 2000, 3000), dropout=0.2, schedule="cosine")
print(f"{frac:.0%} of the training text ({len(split['fit']):,} characters), seed {seed}: held-out bits per byte at "
      + " / ".join(f"{s:,}: {v:.3f}" for s, v in curve.items()) + f" steps ({passes:.0f} passes)")

In [ ]:
for frac in (0.25, 0.5):
    for seed in (0, 1):
        %run datafrac.py {frac} {seed}

The post printed:

```text
25% of the training text (148,404 characters), seed 0: held-out bits per byte at 250: 3.144 / 500: 2.979 / 750: 3.142 / 1,000: 3.308 / 2,000: 4.228 / 3,000: 4.878 / 4,000: 5.227 steps (221 passes)
25% of the training text (148,404 characters), seed 1: held-out bits per byte at 250: 3.142 / 500: 3.010 / 750: 3.212 / 1,000: 3.373 / 2,000: 4.178 / 3,000: 4.842 / 4,000: 5.216 steps (221 passes)
50% of the training text (296,808 characters), seed 0: held-out bits per byte at 250: 3.132 / 500: 2.869 / 750: 2.968 / 1,000: 2.916 / 2,000: 3.312 / 3,000: 3.535 / 4,000: 3.726 steps (110 passes)
50% of the training text (296,808 characters), seed 1: held-out bits per byte at 250: 3.168 / 500: 2.826 / 750: 2.915 / 1,000: 2.983 / 2,000: 3.080 / 3,000: 3.444 / 4,000: 3.608 steps (110 passes)
```

## What the GPU is doing

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/tiny-gpt-from-scratch/#what-the-gpu-is-doing)

In [ ]:
import time
import torch
import torch.nn.functional as F
from gpt import GPT, Config
from train import final_split, tokenizer_for
GPU = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"  # notebook: the post ran on "mps"

# How much of the GPU a training step uses. Model FLOPs per token: 6 per
# parameter (6ND, from "Backpropagation, by Hand") plus attention's own work, 12 L T D (scores and
# weighted sums, forward and backward). Compared with the FP32 matmul peak
# measured on this machine in "How a GPU Runs Deep Learning": 2,878 GFLOP/s.
PEAK = 2878e9
for kind in ("char", "bpe"):
    V = tokenizer_for(kind, final_split["fit"])[0]
    c = Config(V=V)
    torch.manual_seed(0)
    m = GPT(c).to(GPU)
    opt = torch.optim.AdamW(m.parameters(), 1e-3)
    B = 32
    x = torch.randint(0, V, (B, c.T + 1), device=GPU)
    n_params = sum(p.numel() for p in m.parameters())
    for i in range(25):
        if i == 5:
            torch.mps.synchronize(); t0 = time.perf_counter()
        loss = F.cross_entropy(m(x[:, :-1]).reshape(-1, V), x[:, 1:].reshape(-1))
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    torch.mps.synchronize()
    step = (time.perf_counter() - t0) / 20
    tokens = B * c.T
    flops = tokens * (6 * n_params + 12 * c.L * c.T * c.D)
    print(f"{kind:4s}: {n_params:,} parameters, {step * 1e3:.0f} ms per step of {tokens:,} tokens, "
          f"{tokens / step:,.0f} tokens/s, {flops / step / 1e9:.0f} GFLOP/s, MFU {flops / step / PEAK:.1%}")

The post printed:

```text
char: 1,799,616 parameters, 149 ms per step of 8,192 tokens, 54,915 tokens/s, 723 GFLOP/s, MFU 25.1%
bpe : 2,164,416 parameters, 161 ms per step of 8,192 tokens, 50,726 tokens/s, 778 GFLOP/s, MFU 27.0%
```

In [ ]:
import time
import torch
import torch.nn.functional as F
from gpt import GPT, Config
GPU = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"  # notebook: the post ran on "mps"

# Where a training step's 150 ms go: the model's matrix multiplies alone,
# at the step's real shapes (8,192 rows), forward and backward; attention's
# fused call alone, with and without dropout; and AdamW's update.
dev = GPU
def timed(fn, n=20):
    for _ in range(3):
        fn()
    torch.mps.synchronize(); t0 = time.perf_counter()
    for _ in range(n):
        fn()
    torch.mps.synchronize()
    return (time.perf_counter() - t0) / n * 1e3

c = Config(V=148)
M, D, F_ = 32 * c.T, c.D, 512
shapes = [(D, 3 * D), (D, D), (D, F_), (D, F_), (F_, D)] * c.L + [(D, c.V)]   # per layer, then unembedding
xs = [torch.randn(M, k, device=dev, requires_grad=True) for k, _ in shapes]
ws = [torch.randn(k, n, device=dev, requires_grad=True) for k, n in shapes]
def matmuls():
    sum((x @ w).sum() for x, w in zip(xs, ws)).backward()
flops = sum(6 * M * k * n for k, n in shapes)                    # 2 forward + 4 backward per multiply-add
t = timed(matmuls)
print(f"the {len(shapes)} matrix multiplies alone, forward and backward: {t:.0f} ms, {flops / t / 1e6:.0f} GFLOP/s")

q, k, v = (torch.randn(32, c.N, c.T, D // c.N, device=dev, requires_grad=True) for _ in range(3))
for p in (0.0, 0.2):
    def attn():
        F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=p).sum().backward()
    print(f"attention's fused call, {c.L} layers, dropout {p}: {c.L * timed(attn):.0f} ms")

m = GPT(c).to(dev)
opt = torch.optim.AdamW(m.parameters(), 1e-3)
for p in m.parameters():
    p.grad = torch.zeros_like(p)
print(f"AdamW's update of all parameters: {timed(opt.step):.1f} ms")

The post printed:

```text
the 21 matrix multiplies alone, forward and backward: 45 ms, 1960 GFLOP/s
attention's fused call, 4 layers, dropout 0.0: 40 ms
attention's fused call, 4 layers, dropout 0.2: 53 ms
AdamW's update of all parameters: 1.5 ms
```

## What it writes, and what it copied

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/tiny-gpt-from-scratch/#what-it-writes-and-what-it-copied)

In [ ]:
%%writefile generate.py
import sys
import torch
from gpt import GPT, Config

# Sample from the saved model at several checkpoints: feed the last T
# characters, take the distribution for the next one, sample, repeat. Then
# check memorization: the longest stretch of each sample that appears word
# for word in the training posts.
train = open("blog_train.txt").read()

def longest_copied(text):
    best, i = "", 0
    for i in range(len(text)):
        j = i + len(best) + 1
        while j <= len(text) and text[i:j] in train:
            best, j = text[i:j], j + 1
    return best

@torch.no_grad()
def sample(path, prompt, n=300, temperature=0.8, seed=0, greedy=False):
    ckpt = torch.load(path, weights_only=False)
    c = Config(**ckpt["config"])
    model = GPT(c); model.load_state_dict(ckpt["model"]); model.eval()
    vocab = ckpt["vocab"]
    index = {ch: i + 1 for i, ch in enumerate(vocab)}
    g = torch.Generator().manual_seed(seed)
    ids = [index.get(ch, 0) for ch in prompt]
    for _ in range(n):
        logits = model(torch.tensor([ids[-c.T:]]))[0, -1] / temperature
        ids.append(logits.argmax().item() if greedy else
                   torch.multinomial(torch.softmax(logits, -1), 1, generator=g).item())
    return "".join(vocab[i - 1] if i > 0 else "?" for i in ids)

prompt = sys.argv[1] if len(sys.argv) > 1 else "The KV cache"
for step in (250, 1000, 4000):
    text = sample(f"gpt_{step}.pt", prompt)
    copied = longest_copied(text[len(prompt):])
    print(f"--- after {step:,} steps (temperature 0.8)\n{text}\n"
          f"[longest stretch copied from the training posts: {len(copied)} characters: {copied!r}]\n")

# For comparison: text a person wrote, never trained on. Three 300-character
# pieces of the held-out posts, and their longest stretch found in training.
held_out = open("blog_val.txt").read()
for start in (10_000, 25_000, 40_000):
    copied = longest_copied(held_out[start:start + 300])
    print(f"held-out text from character {start:,}: longest stretch also in the training posts "
          f"{len(copied)} characters: {copied!r}")

# Ten more samples from the same prompt, and greedy decoding (always the
# most likely next character) from a prompt that looks like the start of
# one of the blog's code blocks.
lengths = [len(longest_copied(sample("gpt_4000.pt", prompt, seed=s)[len(prompt):])) for s in range(1, 11)]
print(f"\nten more samples at 4,000 steps: longest copied stretch {min(lengths)} to {max(lengths)} characters")
code = "```python\nimport torch\n"
text = sample("gpt_4000.pt", code, n=200, greedy=True)
copied = longest_copied(text[len(code):])
print(f"greedy, from a code prompt:\n{text}\n[longest stretch copied: {len(copied)} characters: {copied!r}]")

In [ ]:
%run generate.py

The post printed:

```text
--- after 250 steps (temperature 0.8)
The KV cache an the rownumes the Cone
the this
***Kuad models is small in the is the methat comput, a abl.
- **The wult it's approduction the rough with recodes log-own is so
  matches the model's every over *attention the loss
blocks is every the the revery in that training, un every eform in model are math mo
[longest stretch copied from the training posts: 14 characters: "es the model's"]

--- after 1,000 steps (temperature 0.8)
The KV cache context of units to Context at it
sequence one scheduler be has is all attention 0.1. The blog loss context
process the one He in the whole recosystem to computed to the $W_V$, the
with the other attention catch from the three with the one number of $\matrix 1$.

The gradient. The model can move mo
[longest stretch copied from the training posts: 17 characters: ' from the three w']

--- after 4,000 steps (temperature 0.8)
The KV cache post
1's own exactly it can always a KV cache per layer, because it can be what
predicts above a fixed-size process that keeps the fixed prompt per log-odds
per-token context sequences and doesn't change, and dividing this post's
change in the next term training, use the serving posts are predictin
[longest stretch copied from the training posts: 19 characters: ' the serving posts ']

held-out text from character 10,000: longest stretch also in the training posts 27 characters: 's |\n| --- | --- | --- |\n| `'
held-out text from character 25,000: longest stretch also in the training posts 17 characters: '.\n\n### Where the '
held-out text from character 40,000: longest stretch also in the training posts 70 characters: ' = \\operatorname{softmax}\\!\\left(\\frac{QK^\\top}{\\sqrt{d_k}} + M\\right)'

ten more samples at 4,000 steps: longest copied stretch 15 to 38 characters
greedy, from a code prompt:
```python
import torch
from router_data import split, featurize
from overfit import sigmoid, log_loss, train

# AdamW that starts the same per-step, the same per-sequence sequence length
# structured and context problem tha
[longest stretch copied: 89 characters: 'from router_data import split, featurize\nfrom overfit import sigmoid, log_loss, train\n\n# ']
```